# Chapitre 1.9 : le faire raisonner, les traces et le GRPO

Après l'alignement du chapitre 1.8, « ton IA » répond mieux. Mais un problème de calcul ne se devine pas : il faut le poser, le décomposer, puis donner un nombre, et ce nombre est **juste ou faux**. Dans ce carnet, tu regardes **un problème et sa trace** (une étape de calcul par ligne), tu **vérifies une réponse** comme le fait la forge, tu refais **à la main l'avantage** de chaque réponse d'un groupe, tu vois **pourquoi un groupe où tout le monde a la même note n'apprend rien**, puis tu fais un **tout petit GRPO** sur un modèle minuscule.

Ce carnet accompagne [la page du chapitre 1.9](https://chipat-neko.github.io/forge-ia/partie-1/1-9/) : tu peux la lire avant, ou en même temps.

**Ce que tu vas faire**

1. regarder un problème généré, sa trace et la conversation que « ton IA » lit en jetons ;
2. vérifier plusieurs réponses à ce problème, avec la règle de la forge ;
3. calculer l'avantage de chaque réponse d'un groupe, et écrire toi-même cette fonction ;
4. calculer la chance qu'un groupe de réponses n'apprenne rien ;
5. regarder un petit GRPO, avec un modèle minuscule qui apprend à répéter un chiffre.

**Comment l'utiliser**

Lance les cellules de code de haut en bas : clique sur le bouton rond avec un triangle, à gauche de la cellule, ou tape Maj + Entrée. Tu peux aussi choisir « Exécution », puis « Tout exécuter ». Tu peux modifier le code et le relancer autant de fois que tu veux : tu ne peux rien casser.

Ce carnet calcule sur le processeur : le modèle de l'étape 5 est minuscule, le processeur suffit. Les nombres de l'étape 5 peuvent varier un peu d'une machine à l'autre : ce que le carnet affirme, il le calcule et l'imprime. Il ne télécharge aucune version de « ton IA ».

## Étape 0 : récupérer le code

Le code du projet est dans un dépôt GitHub public. La cellule suivante le télécharge avec `git clone`, installe la petite bibliothèque `regex` dont le tokenizer a besoin, puis indique à Python où trouver le projet et la forge.

In [ ]:
import os
import sys
from pathlib import Path

# Sur Colab, on télécharge le dépôt public. Quand ce carnet est lancé par les tests
# du projet, la variable FORGE_IA_LOCAL donne le chemin d’un dépôt déjà présent :
# on l’utilise, sans rien télécharger ni installer.
depot_local = os.environ.get("FORGE_IA_LOCAL")
if depot_local:
    depot = Path(depot_local)
else:
    depot = Path("forge-ia")
    if not depot.exists():
        !git clone --depth 1 https://github.com/chipat-neko/forge-ia
    !pip install --quiet regex

projet = depot / "projets" / "1.9-raisonnement"
if not projet.exists():
    raise FileNotFoundError(
        f"Le dossier {projet} est introuvable. Si le dépôt forge-ia était déjà là, "
        "supprime-le (panneau des fichiers, à gauche), puis relance cette cellule\u202f; sinon, vérifie ta connexion."
    )

# Python cherche les modules dans les dossiers de sys.path : on y ajoute ce projet, et forge.
sys.path.insert(0, str(projet.resolve()))
sys.path.insert(0, str((depot / "projets" / "forge").resolve()))

import itertools
import json

import torch
from forge.raisonnement import grpo
from forge.conversation.format import Reponse
from forge.raisonnement.verification import est_juste
from forge.raisonnement.problemes import generer_probleme

from raisonnement_pas_a_pas.avantages import avantages_a_la_main, sans_signal
from raisonnement_pas_a_pas.petit_grpo import essai_grpo
from raisonnement_pas_a_pas.problemes import montrer_probleme, verifier_reponse

appareil = "cpu"  # le modèle est minuscule : le processeur suffit


def nombre(x, decimales=0):
    # À la française : une espace fine entre les milliers, une virgule pour les décimales.
    return f"{x:,.{decimales}f}".replace(",", "\u202f").replace(".", ",")


print(f"Ce carnet calcule sur le processeur, avec PyTorch {torch.__version__.split('+')[0]}.")

## Étape 1 : un problème, et sa trace

« Ton IA » apprend à raisonner sur des **problèmes de calcul en français**, plus faciles que ceux d'une classe de collège. Chacun est **généré par programme** à partir d'une graine : la même graine donne toujours le même problème, avec sa **réponse** et sa **trace**, une étape de calcul par ligne (`a + b = c`). La réponse n'est jamais devinée, elle est calculée en même temps que l'énoncé s'écrit, donc elle est juste par construction.

Pour apprendre à raisonner, « ton IA » lit des conversations où l'utilisateur dit l'énoncé et où elle **pense** la trace, puis répond « La réponse est N. ». La perte ne porte que sur ce que dit l'assistant (sa pensée et sa réponse), jamais sur l'énoncé, comme au chapitre 1.7.

La cellule montre un problème, vérifie sa trace étape par étape, et montre la conversation en jetons.

In [ ]:
# À toi : change la famille, la difficulté ou la graine, puis relance la cellule.
# Familles : addition, soustraction, multiplication, partage, achats, ages (chacune a ses difficultés).
famille, difficulte, graine = "achats", 2, 7
p = montrer_probleme(famille, difficulte, graine)

print(f"Problème {p['id']}\u202f:")
print(f"  {p['enonce']}")
print("Sa trace, une étape de calcul par ligne\u202f:")
for etape in p["etapes"]:
    print(f"  {etape}")
print(f"La réponse est {p['reponse']}.")

# Le code vérifie la trace, au lieu de te demander de le croire.
operations = {"+": lambda a, b: a + b, "−": lambda a, b: a - b, "×": lambda a, b: a * b, "÷": lambda a, b: a // b}
etapes_justes = []
for etape in p["etapes"]:
    gauche, _, resultat = etape.partition(" = ")
    a, operation, b = gauche.split(" ")
    etapes_justes.append(operations[operation](int(a), int(b)) == int(resultat))
termine_sur_la_reponse = p["etapes"][-1].endswith(f" = {p['reponse']}")
print(f"Chaque étape est un calcul juste\u202f: {all(etapes_justes)}. La dernière étape donne la réponse\u202f: {termine_sur_la_reponse}.")

# La conversation que « ton IA » lit, en jetons. Un ▶ marque ceux que la perte apprend.
c = p["conversation"]
print()
print(f"La conversation fait {len(c['ids'])} jetons, dont {sum(c['appris'])} sont appris.")
for piece, appris in list(zip(c["pieces"], c["appris"]))[:24]:
    print(f"  {'▶' if appris else ' '} {piece!r}")
if len(c["ids"]) > 24:
    print(f"  … et {len(c['ids']) - 24} jetons de plus.")
debut_assistant = c["pieces"].index("<|assistant|>")
enonce_jamais_appris = not any(c["appris"][: debut_assistant + 1])
pensee_et_reponse_apprises = all(c["appris"][debut_assistant + 1 :])
print(f"L’énoncé n’est jamais appris\u202f: {enonce_jamais_appris}. La pensée et la réponse le sont\u202f: {pensee_et_reponse_apprises}.")

## Étape 2 : vérifier une réponse

Comment savoir si une réponse est juste, sans juge ? La forge applique une règle simple, qu'un programme vérifie en une ligne :

- elle lit le **dernier nombre** du texte de la réponse (les milliers séparés par une espace, la virgule décimale, le signe moins) ;
- la réponse doit être **terminée** : une réponse coupée par la limite de jetons n'est jamais juste, même si son dernier nombre est le bon, car rien ne dit qu'il est complet (« 12 » peut venir de « 120 ») ;
- la **pensée** ne compte pas, seul le texte qui suit.

La cellule écrit plusieurs réponses possibles au problème de l'étape 1, et dit pour chacune le nombre lu et le verdict. Elle vérifie aussi que ce verdict est celui de la forge (`est_juste`).

In [ ]:
# À toi : change ou ajoute des réponses (le texte, et si elle a fini son tour), puis relance la cellule.
bonne = p["reponse"]
reponses = [
    (f"La réponse est {bonne}.", True),
    (f"{bonne}", True),
    (f"Je trouve {bonne + 1}.", True),
    (f"Je trouve d’abord {bonne + 10}, puis {bonne}.", True),
    (f"Je trouve d’abord {bonne}, puis {bonne + 10}.", True),
    (f"La réponse est {bonne}", False),
    ("Je ne sais pas.", True),
    (f"Environ {bonne + 1}, soit exactement {bonne}.", True),
]
meme_verdict = True
for texte, terminee in reponses:
    verdict = verifier_reponse(texte, terminee, bonne)
    de_la_forge = est_juste(Reponse(texte=texte, pensee=None, ids=[], terminee=terminee), bonne)
    meme_verdict = meme_verdict and verdict["juste"] == de_la_forge
    lu = "aucun nombre" if verdict["lu"] is None else verdict["lu"]
    print(f"  {'juste' if verdict['juste'] else 'fausse':<6} lu {lu:<12} {'' if terminee else '(coupée) '}« {texte} »")
print(f"Même verdict que la forge\u202f: {meme_verdict}")

## Étape 3 : l'avantage d'une réponse dans son groupe

Le **GRPO** (« Group Relative Policy Optimization ») ne demande à « ton IA » qu'une chose : **plusieurs réponses au même problème**, un **groupe**. Chaque réponse reçoit une **récompense**, ici 1 si elle est juste et 0 sinon. Son **avantage** est sa récompense **moins la moyenne de son groupe** : positif si elle est meilleure que ses sœurs, négatif si elle est pire.

```
avantage = récompense − moyenne des récompenses du groupe
```

La perte du GRPO est une somme d'avantages fois des log-probabilités des jetons écrits : elle pousse vers le haut les réponses meilleures que la moyenne, et vers le bas les autres. Sans division par l'écart-type, sans comparaison à une version de référence.

La cellule prend les réponses de l'étape 2 comme un groupe, calcule leur avantage à la main (`avantages_a_la_main`) et le compare à celui de la forge (`grpo.avantages`), calculé sur des tenseurs en float64.

In [ ]:
recompenses = [1.0 if verifier_reponse(texte, terminee, bonne)["juste"] else 0.0 for texte, terminee in reponses]
avantages = avantages_a_la_main([recompenses])[0]
moyenne = sum(recompenses) / len(recompenses)
print(f"Récompenses du groupe\u202f: {recompenses}")
print(f"Moyenne du groupe\u202f: {nombre(moyenne, 3)}")
for (texte, terminee), r, a in zip(reponses, recompenses, avantages):
    print(f"  récompense {r:.0f}, avantage {nombre(a, 3):>6}  « {texte} »")

de_la_forge = grpo.avantages(torch.tensor([recompenses], dtype=torch.float64))[0].tolist()
egaux = all(abs(x - y) <= 1e-12 for x, y in zip(avantages, de_la_forge))
print(f"À la main et dans la forge, à 10⁻¹² près\u202f: {egaux}")
print(f"Les avantages somment à zéro\u202f: {abs(sum(avantages)) <= 1e-12}")

# Un groupe où toutes les réponses ont la même récompense.
for nom, groupe in (("toutes justes", [1.0] * 8), ("toutes fausses", [0.0] * 8)):
    print(f"Un groupe de {len(groupe)} réponses {nom}\u202f: avantages {avantages_a_la_main([groupe])[0]}")

## Exercice : écrire l'avantage d'un groupe

Écris une fonction `avantage(recompenses)` qui rend, pour une liste de récompenses (celles d'un groupe), la liste des avantages : la récompense de chaque réponse moins la moyenne du groupe.

La cellule compare ta fonction à `avantages_a_la_main` sur six groupes.

In [ ]:
def avantage(recompenses):
    # À toi : renvoie la liste des avantages. Remplace la ligne suivante.
    return []


groupes = [
    [1.0, 0.0, 0.0, 1.0],
    [0.0, 0.0, 0.0, 0.0],
    [1.0, 1.0, 1.0, 1.0],
    [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    [0.5, 0.25, 0.75],
    [1.0, 0.0],
]
reussi = True
for groupe in groupes:
    attendu = avantages_a_la_main([groupe])[0]
    obtenu = avantage(groupe)
    ok = len(obtenu) == len(attendu) and all(abs(x - y) <= 1e-12 for x, y in zip(obtenu, attendu))
    reussi = reussi and ok
    print(f"{str(groupe):<44} ta fonction {str([round(x, 3) for x in obtenu]):<40} {'ok' if ok else 'différent'}")
if reussi:
    print("Défi réussi, bravo.")
else:
    print("Pas encore. Calcule d’abord la moyenne du groupe, puis retire-la à chaque récompense.")

<details>
<summary>Voir une correction (essaie d’abord !)</summary>

La moyenne d'un groupe est la somme de ses récompenses divisée par leur nombre. L'avantage de chaque réponse est sa récompense moins cette moyenne.

Remplace la fonction de la cellule de l'exercice par :

```python
def avantage(recompenses):
    moyenne = sum(recompenses) / len(recompenses)
    return [r - moyenne for r in recompenses]
```

</details>

## Étape 4 : pourquoi un groupe sans signal n'apprend rien

Si les réponses d'un groupe ont toutes **la même récompense** (toutes justes, ou toutes fausses), leur moyenne est cette récompense, et **tous leurs avantages valent zéro**. La perte de ce groupe est une somme de zéros : il n'ajoute aucun gradient, il n'apprend rien. C'est le cas courant au début de l'entraînement, quand « ton IA » ne réussit presque jamais.

Si chaque réponse est juste avec la chance `p`, un groupe de `n` réponses est sans signal avec la chance

```
pⁿ + (1 − p)ⁿ      (toutes justes, ou toutes fausses)
```

La cellule écrit un petit tableau de cette chance, et vérifie la formule en **dénombrant** tous les groupes possibles d'une taille donnée : la somme des chances de ceux dont les avantages sont tous nuls doit être la même.

In [ ]:
chances = (0.02, 0.05, 0.1, 0.25, 0.5, 0.9)
tailles = (2, 4, 8, 16)
print("Chance qu'un groupe soit sans signal (en %), selon la chance p d'une réponse juste et la taille n du groupe\u202f:")
print(f"  {'p':>5}" + "".join(f"  n = {n:<4}" for n in tailles))
for p_juste in chances:
    print(f"  {nombre(p_juste, 2):>5}" + "".join(f"  {nombre(100 * sans_signal(p_juste, n), 1):>8}" for n in tailles))

# La formule contre le dénombrement des groupes possibles (n petit, pour que la liste reste courte).
p_juste, n = 0.25, 6
somme = 0.0
for groupe in itertools.product((0.0, 1.0), repeat=n):
    chance = 1.0
    for r in groupe:
        chance *= p_juste if r == 1.0 else 1 - p_juste
    if all(a == 0 for a in avantages_a_la_main([list(groupe)])[0]):
        somme += chance
print()
print(f"p = {nombre(p_juste, 2)}, n = {n}\u202f: par dénombrement des {2 ** n} groupes possibles {nombre(somme, 6)}, par la formule {nombre(sans_signal(p_juste, n), 6)}.")
print(f"Égaux à 10⁻¹² près\u202f: {abs(somme - sans_signal(p_juste, n)) <= 1e-12}")

# Si « ton IA » ne réussit jamais, aucun groupe ne donne de signal.
taille = max(tailles)
print(f"Si p = 0, un groupe de {taille} réponses est sans signal avec la chance {nombre(sans_signal(0.0, taille), 2)}\u202f: le RL n’a rien à amplifier.")

## Étape 5 : un petit GRPO

Le RL **n'amplifie qu'un comportement déjà présent**. S'il n'y a aucune réponse juste, il n'y a aucun signal, et la forge refuse même de le lancer (son « contrôle de départ » compte d'abord les réponses justes de la version de départ).

La cellule fait l'expérience en petit, sur le processeur, avec le vrai code de la forge :

1. un modèle **minuscule** est d'abord **affiné à répéter un chiffre** : « Répète : 7 » doit donner « 7 » ;
2. il fait ensuite des **pas de GRPO** (`pas_grpo` de la forge) : pour chaque chiffre d'un lot, plusieurs réponses, la récompense 1 si la réponse est juste, l'avantage `r − moyenne` du groupe, puis un pas.

Elle écrit d'abord les réglages de l'essai (la taille du modèle, le nombre de réponses par chiffre), puis, un pas sur deux, la récompense moyenne et la part des groupes sans signal. L'affinage de départ s'arrête **exprès à mi-chemin** de son apprentissage : un modèle presque toujours juste ou presque jamais juste ne donnerait que des groupes sans signal, donc rien à regarder. Rien ne garantit une hausse de la récompense : peu de pas sur un modèle minuscule ne garantissent rien, et le code ne triche pas.

In [ ]:
pas_de_grpo = 20  # change-le, puis relance la cellule
resultat = essai_grpo(appareil, pas=pas_de_grpo)

g = resultat["reglages"]
print(f"Le petit GRPO a tourné sur {resultat['appareil']}, {resultat['pas']} pas, en {nombre(resultat['secondes'], 1)}\u202fs.")
print(f"Réglages\u202f: modèle de {g['couches']} couches et de vecteurs de {g['dim']} nombres, affiné {g['pas_d_affinage']} pas ; "
      f"à chaque pas {g['invites_par_pas']} chiffres × {g['reponses_par_invite']} réponses de {g['jetons_max_reponse']} jetons au plus, taux {nombre(g['taux'], 4)}.")
print(f"Le modèle répète bien {nombre(100 * resultat['reussite_avant'])}\u202f% des dix chiffres avant le GRPO (en glouton), {nombre(100 * resultat['reussite_apres'])}\u202f% après.")
print()
print(f"  {'pas':>4}  {'récompense':>10}  {'groupes sans signal':>19}")
for numero in range(0, resultat["pas"], 2):
    print(f"  {numero + 1:>4}  {nombre(resultat['recompense_moyenne'][numero], 3):>10}  {nombre(resultat['groupes_sans_signal'][numero], 3):>19}")

recompenses_du_pas = resultat["recompense_moyenne"]
signaux_du_pas = resultat["groupes_sans_signal"]
cinq = min(5, len(recompenses_du_pas))
debut_r, fin_r = sum(recompenses_du_pas[:cinq]) / cinq, sum(recompenses_du_pas[-cinq:]) / cinq
debut_s, fin_s = sum(signaux_du_pas[:cinq]) / cinq, sum(signaux_du_pas[-cinq:]) / cinq
print()
print(f"Récompense moyenne des {cinq} premiers pas\u202f: {nombre(debut_r, 3)}, des {cinq} derniers\u202f: {nombre(fin_r, 3)}. Elle a monté\u202f: {'oui' if fin_r > debut_r else 'non'}.")
print(f"Part de groupes sans signal, début\u202f: {nombre(debut_s, 3)}, fin\u202f: {nombre(fin_s, 3)}. Elle a baissé\u202f: {'oui' if fin_s < debut_s else 'non'}.")
avant, apres = resultat["reussite_avant"], resultat["reussite_apres"]
sens = "monté" if apres > avant else "baissé" if apres < avant else "pas bougé"
print(f"La réussite en glouton a {sens} (de {nombre(100 * avant)}\u202f% à {nombre(100 * apres)}\u202f%).")
sans_aucun_signal = sum(1 for s in signaux_du_pas if s == 1.0)
print(f"Pas où aucun groupe n’avait de signal (le pas ne bouge pas les poids)\u202f: {sans_aucun_signal} sur {len(signaux_du_pas)}.")

Lis les verdicts que le code vient de calculer, sans te fier à ce que tu attends : ils valent pour ce modèle, ce lot de chiffres et ces réglages, et pas pour « ton IA ». Ce qui tient dans tous les cas : un pas où tous les groupes sont sans signal ne change aucun poids, et plus la part de groupes sans signal est grande, moins le pas apprend. Ne confonds pas les deux mesures : la **récompense** se mesure sur des réponses **tirées au hasard** (température 1), la **réussite en glouton** sur la réponse **la plus probable** de chaque chiffre. Elles peuvent aller en sens contraire, et la seconde ne porte que sur dix chiffres, donc un seul chiffre de plus ou de moins la fait bouger de dix points. La page du chapitre montre ce qui s'est vraiment passé sur « ton IA », avec le contrôle de départ et les courbes de l'entraînement.

## Dernière étape : le même résultat que la page ?

La page du site n'utilise pas Python : elle refait quatre calculs en TypeScript, directement dans ton navigateur, le **générateur de problèmes**, la **lecture du dernier nombre** d'une réponse, l'**avantage** d'un groupe et la **chance d'un groupe sans signal**. Pour être sûr que les deux donnent les mêmes résultats, le dépôt garde un fichier de référence, `tests/attendus.json`, calculé une fois pour toutes. Les tests de la page relisent ce fichier. La cellule suivante le relit aussi, et le compare à ce que le code de ce carnet vient de calculer.

In [ ]:
attendus = json.loads((projet / "tests" / "attendus.json").read_text(encoding="utf-8"))

# Chaque problème du fichier est-il celui que le code génère ?
problemes_differents = []
for cas in attendus["problemes"] + attendus["problemes_reserves"]:
    refait = montrer_probleme(cas["famille"], cas["difficulte"], cas["graine"])
    if (refait["enonce"], refait["etapes"], refait["reponse"]) != (cas["enonce"], cas["etapes"], cas["reponse"]):
        problemes_differents.append((cas["famille"], cas["difficulte"], cas["graine"]))

# Chaque texte donne-t-il le même nombre lu, et les mêmes verdicts ?
textes_differents = []
for cas in attendus["nombres"]:
    refait = verifier_reponse(cas["texte"], True, 0)["lu"]
    verdicts = [
        (v["attendu"], v["terminee"], verifier_reponse(cas["texte"], v["terminee"], v["attendu"])["juste"]) for v in cas["verdicts"]
    ]
    if refait != cas["lu"] or verdicts != [(v["attendu"], v["terminee"], v["juste"]) for v in cas["verdicts"]]:
        textes_differents.append(cas["texte"])

# Chaque groupe et chaque point de la grille ?
groupes_differents = [g["recompenses"] for g in attendus["avantages"] if avantages_a_la_main([g["recompenses"]])[0] != g["avantages"]]
points_differents = [(g["p"], g["n"]) for g in attendus["sans_signal"] if sans_signal(g["p"], g["n"]) != g["chance"]]

nb_problemes = len(attendus["problemes"]) + len(attendus["problemes_reserves"])
print(f"{nb_problemes} problèmes, {len(attendus['nombres'])} textes, {len(attendus['avantages'])} groupes et {len(attendus['sans_signal'])} points de la grille comparés.")
for cas in problemes_differents[:3]:
    print(f"Problème différent\u202f: {cas}.")
for cas in textes_differents[:3]:
    print(f"Texte différent\u202f: {cas!r}.")
for cas in groupes_differents[:3]:
    print(f"Groupe différent\u202f: {cas}.")
for cas in points_differents[:3]:
    print(f"Point différent\u202f: p, n = {cas}.")

identique = not (problemes_differents or textes_differents or groupes_differents or points_differents)
print(f"Même résultat que la page du site\u202f: {'oui' if identique else 'non'}")  # l’espace avant les deux-points est une espace fine insécable

In [ ]:
print("Ce carnet a tourné sur le processeur.")

## Pour aller plus loin

- Le code complet du projet, avec ses tests : [projets/1.9-raisonnement sur GitHub](https://github.com/chipat-neko/forge-ia/tree/main/projets/1.9-raisonnement).
- La page du chapitre : [chapitre 1.9](https://chipat-neko.github.io/forge-ia/partie-1/1-9/).
- Depuis le dossier du projet, `uv run --extra cpu python -m raisonnement_pas_a_pas probleme achats 2 7` montre un problème, `verifier` vérifie une réponse, `avantages` refait l'avantage d'un groupe, `signal` donne la chance d'un groupe sans signal, et `essai` refait l'étape 5.

Le code de ce carnet est sous licence MIT, ses textes sous licence CC BY 4.0.